# Preprocessing: Before & After Illumination Correction

Demonstrates the illumination correction and background subtraction pipeline on actual TMEM106B imaging data.

**Key functions:**
- `calculate_ic_field()` — compute IC field from a set of images (per-well or per-plate)
- `apply_ic_field()` — divide image by IC field to normalize illumination
- `subtract_background()` — rolling ball background subtraction
- `preprocess_image()` — combined pipeline (IC + optional background subtraction)

In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from tmem_align.io import read_image
from tmem_align.preprocess import (
    calculate_ic_field,
    apply_ic_field,
    subtract_background,
    preprocess_image,
)

## 1. Load Data

Load ND2 images from a single timepoint. We use a subset of wells to calculate the IC field.

In [ ]:
data_dir = Path("/Users/pmihack/claire/tmem_2026/data/260213_Feb16recopy_HYdiff_landingpadlines_survival_384well1/20260228_090815_318")
nd2_files = sorted(data_dir.glob("*.nd2"))
print(f"Total ND2 files in timepoint: {len(nd2_files)}")

# Load a test image
test_file = [f for f in nd2_files if "WellE05" in f.name][0]
test_img = read_image(test_file)
print(f"Test image shape: {test_img.shape}, dtype: {test_img.dtype}")
print(f"Channels: 405nm (DAPI), 561nm (mCherry), 488nm (GFP)")

## 2. Calculate Illumination Correction Field

Using 50 randomly sampled wells from this plate to compute a plate-level IC field.

In [ ]:
# Calculate plate-level IC field from a sample of wells
ic_field = calculate_ic_field(
    [str(f) for f in nd2_files],
    sample_fraction=0.25,  # use 25% of images
    smooth=None,  # auto-calculate smoothing radius
)
print(f"IC field shape: {ic_field.shape}")
print(f"IC field range: [{ic_field.min():.3f}, {ic_field.max():.3f}]")

## 3. Visualize IC Field

The IC field shows the illumination pattern. Brighter regions in the field indicate areas that were originally brighter (center of FOV typically).

In [ ]:
channel_names = ["405nm (DAPI)", "561nm (mCherry)", "488nm (GFP)"]

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for i, (ax, name) in enumerate(zip(axes, channel_names)):
    if ic_field.ndim == 3:
        im = ax.imshow(ic_field[i], cmap="inferno")
    else:
        im = ax.imshow(ic_field, cmap="inferno")
    ax.set_title(f"IC Field — {name}")
    ax.axis("off")
    plt.colorbar(im, ax=ax, fraction=0.046)
fig.suptitle("Illumination Correction Fields (per channel)", fontsize=14)
plt.tight_layout()
plt.show()

## 4. Before / After: Illumination Correction

Side-by-side comparison of raw vs IC-corrected images for each channel.

In [ ]:
corrected = apply_ic_field(test_img, ic_field)

fig, axes = plt.subplots(3, 2, figsize=(12, 16))
for i, name in enumerate(channel_names):
    raw_ch = test_img[i]
    cor_ch = corrected[i]
    vmin, vmax = np.percentile(raw_ch, [1, 99])

    axes[i, 0].imshow(raw_ch, cmap="gray", vmin=vmin, vmax=vmax)
    axes[i, 0].set_title(f"Raw — {name}")
    axes[i, 0].axis("off")

    axes[i, 1].imshow(cor_ch, cmap="gray", vmin=vmin, vmax=vmax)
    axes[i, 1].set_title(f"IC Corrected — {name}")
    axes[i, 1].axis("off")

fig.suptitle("Before / After Illumination Correction (Well E05, Day 20)", fontsize=14)
plt.tight_layout()
plt.show()

## 5. Intensity Profiles

Line profiles across the image center showing how IC flattens the illumination gradient.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
mid_row = test_img.shape[1] // 2

for i, (ax, name) in enumerate(zip(axes, channel_names)):
    raw_profile = test_img[i, mid_row, :].astype(float)
    cor_profile = corrected[i, mid_row, :].astype(float)

    # Smooth for readability
    from scipy.ndimage import uniform_filter1d
    raw_smooth = uniform_filter1d(raw_profile, 50)
    cor_smooth = uniform_filter1d(cor_profile, 50)

    ax.plot(raw_smooth, label="Raw", alpha=0.8)
    ax.plot(cor_smooth, label="Corrected", alpha=0.8)
    ax.set_title(name)
    ax.set_xlabel("X position (px)")
    ax.set_ylabel("Intensity")
    ax.legend()

fig.suptitle("Horizontal Intensity Profiles (center row, smoothed)", fontsize=14)
plt.tight_layout()
plt.show()

## 6. Background Subtraction

Rolling ball background subtraction removes slowly varying background (autofluorescence, optical artifacts).

In [ ]:
# Apply full pipeline: IC + background subtraction
fully_preprocessed = preprocess_image(test_img, ic_field=ic_field, background_radius=100)

fig, axes = plt.subplots(3, 3, figsize=(16, 16))
for i, name in enumerate(channel_names):
    raw_ch = test_img[i]
    ic_ch = corrected[i]
    full_ch = fully_preprocessed[i]
    vmin, vmax = np.percentile(raw_ch, [1, 99])

    axes[i, 0].imshow(raw_ch, cmap="gray", vmin=vmin, vmax=vmax)
    axes[i, 0].set_title(f"Raw — {name}")
    axes[i, 0].axis("off")

    axes[i, 1].imshow(ic_ch, cmap="gray", vmin=vmin, vmax=vmax)
    axes[i, 1].set_title(f"IC Only — {name}")
    axes[i, 1].axis("off")

    axes[i, 2].imshow(full_ch, cmap="gray", vmin=vmin, vmax=vmax)
    axes[i, 2].set_title(f"IC + BG Sub — {name}")
    axes[i, 2].axis("off")

fig.suptitle("Full Preprocessing Pipeline Comparison", fontsize=14)
plt.tight_layout()
plt.show()

## 7. Summary Statistics

In [ ]:
import pandas as pd

stats = []
for i, name in enumerate(channel_names):
    for label, arr in [("Raw", test_img), ("IC Corrected", corrected), ("IC+BG Sub", fully_preprocessed)]:
        ch = arr[i].astype(float)
        stats.append({
            "Channel": name,
            "Stage": label,
            "Mean": f"{ch.mean():.1f}",
            "Std": f"{ch.std():.1f}",
            "CV%": f"{100 * ch.std() / ch.mean():.1f}" if ch.mean() > 0 else "N/A",
            "Min": int(ch.min()),
            "Max": int(ch.max()),
        })

df = pd.DataFrame(stats)
df